# 06 — Behavioral v2: Isolation Forest + Autoencoder retrain (Option A, Phase 3)

Final validation before committing to Option A. Retrains Isolation Forest and Autoencoder on
`data/features/behavioral/cicids2017_behavioral_features_{train,val,test}.parquet` -- the
properly pipeline-integrated output of Phase 1 (`validate_behavioral_join.py`: row-hash join,
dtype-normalized, BENIGN-unmatched-dropped-and-documented) + Phase 2
(`feature_extractor_behavioral.py`: MITRE join, sentinel impute, CTI-encode, MinMaxScaler, capped
SMOTE) -- as opposed to `04_enhanced_ablation_behavioral.ipynb`'s standalone quick prototype,
which read `cicids2017_behavioral_labeled.parquet` directly and built its own ad hoc 60/20/20
split, with no row-hash-join validation and no BENIGN-population accounting.

Random Forest is **not** retrained here -- behavioral features didn't change its known LOAO
weaknesses (a supervised-generalization problem, not a feature-representation one), confirmed in
`05_loao_behavioral.ipynb`. RF stays on `data/features/cicids2017_features_*.parquet`, untouched.

**Conventions reused directly from `01_isolation_forest.ipynb`/`03_autoencoder.ipynb`** (same
architecture, same sweep grids, same per-category recall reporting -- not redesigned):
- Isolation Forest: `n_estimators=100`, starting `contamination=0.20` (the tuned value the
  original 80-feature run converged on), re-swept here against `[0.05, 0.10, 0.15, 0.20]` on the
  new feature set to confirm it's still the best choice rather than assuming it carries over.
- Autoencoder: same architecture (`hidden_dims=[64, 32]`, `bottleneck_dim=16`, ReLU, `lr=1e-3`,
  `epochs=30`, early stopping patience 5), same percentile threshold-sweep grid
  (`[90, 95, 99]` of val-set BENIGN reconstruction error). Uses the manual GPU-resident batching
  from `05_loao_behavioral.ipynb` (an efficiency change only, not a semantic one -- same
  minibatch-SGD behavior, ~4x fewer Python-loop batches) rather than `03_autoencoder.ipynb`'s
  original `DataLoader`, since that's the more refined convention already validated in this repo.
- CTI-encoded columns (`technique_id_encoded`, `tactic_encoded`, `mapping_confidence_encoded`,
  `is_known_technique`) excluded from model input, same precedent as every prior notebook
  (deterministic functions of the true label, not real-world available at detection time).

**Explicitly documented per this phase's requirement**: Autoencoder's real BENIGN training
population is smaller here than the original 80-feature pipeline's, because Phase 1 dropped the
~23.6% of BENIGN rows that didn't hash-match between the two independent CICFlowMeter
distributions. The actual row count (not just "BENIGN-only" as a label) is computed and printed
below, and carried into the saved config.

**Outputs are namespaced to avoid collision** with both the original baseline and the standalone
prototype notebook: `models/isolation_forest_behavioral_v2.pkl`,
`models/autoencoder_behavioral_v2.pt` (+ `_config.json`),
`reports/behavioral_v2_val_metrics.csv`, `reports/behavioral_v2_vs_baseline_comparison.csv`.

## 1. Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import torch
import torch.nn as nn

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("torch version:", torch.__version__, " | cuda available:", torch.cuda.is_available())

torch version: 2.12.1+cu130  | cuda available: True


In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BEHAVIORAL_FEATURES_DIR = ROOT / "data" / "features" / "behavioral"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

print("ROOT:", ROOT)

ROOT: E:\thesis


## 2. Config

`CONTAMINATION` starts at `0.20` (the value the original 80-feature run converged on, not
`'auto'`) since we already know from `01_isolation_forest.ipynb` that `'auto'` underperforms on
this data -- Section 5 below still re-sweeps around it rather than assuming it transfers
unchanged to the new feature set.

In [3]:
RANDOM_STATE = 42

# Isolation Forest
CONTAMINATION = 0.20
N_ESTIMATORS = 100
CONTAMINATION_SWEEP = [0.05, 0.10, 0.15, 0.20]

# Autoencoder -- architecture identical to 03_autoencoder.ipynb
AE_HIDDEN_DIMS = [64, 32]
AE_BOTTLENECK_DIM = 16
AE_ACTIVATION = "relu"
AE_LEARNING_RATE = 1e-3
AE_EPOCHS = 30
AE_BATCH_SIZE = 8192  # manual GPU-resident batching, same convention as 05_loao_behavioral.ipynb
AE_WEIGHT_DECAY = 0.0
AE_EARLY_STOPPING_PATIENCE = 5
BENIGN_INTERNAL_VAL_FRAC = 0.10
THRESHOLD_PERCENTILES = [90, 95, 99]

AE_CONFIG = {
    "hidden_dims": AE_HIDDEN_DIMS,
    "bottleneck_dim": AE_BOTTLENECK_DIM,
    "activation": AE_ACTIVATION,
    "learning_rate": AE_LEARNING_RATE,
    "epochs": AE_EPOCHS,
    "batch_size": AE_BATCH_SIZE,
    "weight_decay": AE_WEIGHT_DECAY,
    "early_stopping_patience": AE_EARLY_STOPPING_PATIENCE,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE: cuda


## 3. Load data

`data/features/behavioral/cicids2017_behavioral_features_{train,val,test}.parquet` -- Phase 1+2
output. `train` is capped-SMOTE-resampled (same formula as `feature_extractor.py`); `val`/`test`
are untouched.

In [4]:
train_df = pd.read_parquet(BEHAVIORAL_FEATURES_DIR / "cicids2017_behavioral_features_train.parquet")
val_df = pd.read_parquet(BEHAVIORAL_FEATURES_DIR / "cicids2017_behavioral_features_val.parquet")
test_df = pd.read_parquet(BEHAVIORAL_FEATURES_DIR / "cicids2017_behavioral_features_test.parquet")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (1483957, 102)
val:   (405085, 102)
test:  (405142, 102)


## 4. Feature selection

Same exclusion precedent as `01_isolation_forest.ipynb`/`03_autoencoder.ipynb`/
`05_loao_behavioral.ipynb`: identifier/label columns and the 4 CTI-encoded columns (deterministic
functions of the true label, reserved for the LLM explanation layer, not any detection model) are
excluded from model input. `orig_row_index` (added in Phase 1 for the eventual Fusion Layer join,
not a real feature) is excluded here too.

In [5]:
NON_FEATURE_COLS = [
    "Label",
    "technique_id", "technique_name", "tactic", "mapping_confidence",
    "technique_id_encoded", "tactic_encoded", "mapping_confidence_encoded", "is_known_technique",
    "orig_row_index",
]

def select_feature_columns(df, non_feature_cols=NON_FEATURE_COLS):
    return [
        c for c in df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

feature_cols = select_feature_columns(train_df)
print(f"Model feature count: {len(feature_cols)} (expected 92 = 90 network incl. 12 behavioral + 2 sentinel-missing-flag)")
assert len(feature_cols) == 92
assert feature_cols == select_feature_columns(val_df) == select_feature_columns(test_df)
feature_cols

Model feature count: 92 (expected 92 = 90 network incl. 12 behavioral + 2 sentinel-missing-flag)


['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

## 5. BENIGN training population -- explicit row count (not just "BENIGN-only")

Phase 1 dropped the ~23.6% of BENIGN rows that didn't hash-match between the two independent
CICFlowMeter distributions. This directly shrinks the Autoencoder's real training population
(it's the model that trains BENIGN-only) -- documented here as an actual count, not a label.

In [6]:
BASELINE_TRAIN_BENIGN_COUNT = 1_257_033  # from CLAUDE.md / 03_autoencoder.ipynb: cicids2017_features_train.parquet BENIGN count

behavioral_train_benign_df = train_df[train_df["Label"] == "BENIGN"]
n_behavioral_benign = len(behavioral_train_benign_df)
pct_reduction = 100 * (1 - n_behavioral_benign / BASELINE_TRAIN_BENIGN_COUNT)

print(f"Original 80-feature pipeline train BENIGN count : {BASELINE_TRAIN_BENIGN_COUNT:,}")
print(f"Behavioral (Phase 1+2) train BENIGN count        : {n_behavioral_benign:,}")
print(f"Reduction                                        : {pct_reduction:.2f}%")
print()
print("This is the actual population the Autoencoder below trains on (BENIGN-only, minus the")
print(f"{BENIGN_INTERNAL_VAL_FRAC:.0%} internal validation split carved out of it next).")

Original 80-feature pipeline train BENIGN count : 1,257,033
Behavioral (Phase 1+2) train BENIGN count        : 960,726
Reduction                                        : 23.57%

This is the actual population the Autoencoder below trains on (BENIGN-only, minus the
10% internal validation split carved out of it next).


## 6. Isolation Forest

### 6.1 Main fit at the starting `contamination=0.20`

In [7]:
X_train = train_df[feature_cols].to_numpy(dtype=np.float64)
X_val = val_df[feature_cols].to_numpy(dtype=np.float64)
y_true = (val_df["Label"] != "BENIGN").astype(int).to_numpy()

iso_forest_main = IsolationForest(
    n_estimators=N_ESTIMATORS, contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1,
)
iso_forest_main.fit(X_train)

raw_pred = iso_forest_main.predict(X_val)
prediction = (raw_pred == -1).astype(int)
anomaly_score = -iso_forest_main.score_samples(X_val)

precision = precision_score(y_true, prediction)
recall = recall_score(y_true, prediction)
f1 = f1_score(y_true, prediction)
auc_roc = roc_auc_score(y_true, anomaly_score)

print(f"contamination={CONTAMINATION}  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}  AUC-ROC={auc_roc:.4f}")

contamination=0.2  Precision=0.5870  Recall=0.5541  F1=0.5700  AUC-ROC=0.8378


### 6.2 Contamination sweep -- confirm `0.20` still wins on the new feature set

Same grid as `01_isolation_forest.ipynb`. AUC-ROC should be identical across all sweep values
(same trees/ranking, only the cutoff differs) -- asserted, not just claimed.

In [8]:
sweep_rows = []
for c in CONTAMINATION_SWEEP:
    m = IsolationForest(n_estimators=N_ESTIMATORS, contamination=c, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train)
    pred_c = (m.predict(X_val) == -1).astype(int)
    score_c = -m.score_samples(X_val)
    sweep_rows.append({
        "contamination": c,
        "offset_": m.offset_,
        "precision": precision_score(y_true, pred_c),
        "recall": recall_score(y_true, pred_c),
        "f1": f1_score(y_true, pred_c),
        "auc_roc": roc_auc_score(y_true, score_c),
    })

contamination_sweep_df = pd.DataFrame(sweep_rows)
assert np.allclose(contamination_sweep_df["auc_roc"], contamination_sweep_df["auc_roc"].iloc[0], atol=1e-9), \
    "AUC-ROC should be identical across contamination values -- same ranking, only the cutoff differs"
print("Confirmed: AUC-ROC is identical across all contamination values (threshold-independent).\n")
contamination_sweep_df

Confirmed: AUC-ROC is identical across all contamination values (threshold-independent).



,contamination,offset_,precision,recall,f1,auc_roc
0,0.05,-0.517927,0.507638,0.109280,0.179845,0.83776
1,0.10,-0.481849,0.641603,0.302145,0.410823,0.83776
2,0.15,-0.456356,0.635859,0.443827,0.522766,0.83776
3,0.20,-0.430553,0.586964,0.554082,0.570049,0.83776


In [9]:
best_idx = contamination_sweep_df["f1"].idxmax()
CONTAMINATION = contamination_sweep_df.loc[best_idx, "contamination"]
print(f"Chosen contamination (best F1 on val): {CONTAMINATION!r}  (F1 = {contamination_sweep_df.loc[best_idx, 'f1']:.4f})")
if CONTAMINATION == 0.20:
    print("Confirms the starting point (0.20, carried over from the 80-feature run) is still optimal on the new feature set.")
else:
    print(f"NOTE: optimal contamination shifted to {CONTAMINATION!r} on the new feature set -- differs from the 80-feature run's 0.20.")

iso_forest_main = IsolationForest(
    n_estimators=N_ESTIMATORS, contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1,
)
iso_forest_main.fit(X_train)

raw_pred = iso_forest_main.predict(X_val)
prediction = (raw_pred == -1).astype(int)
anomaly_score = -iso_forest_main.score_samples(X_val)

if_precision = precision_score(y_true, prediction)
if_recall = recall_score(y_true, prediction)
if_f1 = f1_score(y_true, prediction)
if_auc_roc = roc_auc_score(y_true, anomaly_score)

print(f"\nFinal (contamination={CONTAMINATION!r}) -- Precision {if_precision:.4f}  Recall {if_recall:.4f}  F1 {if_f1:.4f}  AUC-ROC {if_auc_roc:.4f}")

val_df = val_df.copy()
val_df["if_prediction"] = prediction
val_df["if_anomaly_score"] = anomaly_score

Chosen contamination (best F1 on val): np.float64(0.2)  (F1 = 0.5700)
Confirms the starting point (0.20, carried over from the 80-feature run) is still optimal on the new feature set.



Final (contamination=np.float64(0.2)) -- Precision 0.5870  Recall 0.5541  F1 0.5700  AUC-ROC 0.8378


## 6.3 Per-category recall (Isolation Forest)

In [10]:
if_category_recall = (
    val_df[val_df["Label"] != "BENIGN"]
    .groupby("Label")
    .apply(lambda g: pd.Series({"recall": g["if_prediction"].mean(), "n_rows": len(g)}))
    .sort_values("recall")
)
if_category_recall

C:\Users\devMrX\AppData\Local\Temp\ipykernel_17820\2532552207.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"recall": g["if_prediction"].mean(), "n_rows": len(g)}))


,recall,n_rows
Label,,
SSH-Patator,0.000000,644.0
Web Attack - Sql Injection,0.000000,4.0
Web Attack - XSS,0.000000,131.0
FTP-Patator,0.000843,1186.0
PortScan,0.013893,18139.0
Bot,0.023136,389.0
Web Attack - Brute Force,0.054422,294.0
DoS GoldenEye,0.238211,2057.0
DoS Slowhttptest,0.374163,1045.0


## 7. Autoencoder

### 7.1 Architecture + training loop (identical to `03_autoencoder.ipynb`, manual GPU batching per `05_loao_behavioral.ipynb`)

In [11]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, bottleneck_dim, activation="relu"):
        super().__init__()
        act_cls = {"relu": nn.ReLU, "tanh": nn.Tanh, "leaky_relu": nn.LeakyReLU}[activation]

        encoder_dims = [input_dim] + list(hidden_dims) + [bottleneck_dim]
        decoder_dims = [bottleneck_dim] + list(hidden_dims)[::-1] + [input_dim]

        encoder_layers = []
        for i in range(len(encoder_dims) - 1):
            encoder_layers.append(nn.Linear(encoder_dims[i], encoder_dims[i + 1]))
            if i < len(encoder_dims) - 2:
                encoder_layers.append(act_cls())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(act_cls())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def compute_reconstruction_error(model, X, device):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        reconstructed = model(X_tensor)
        errors = torch.mean((reconstructed - X_tensor) ** 2, dim=1).cpu().numpy()
    return errors


def train_autoencoder(X_fit, X_internal_val, input_dim, config, device, verbose=True):
    model = Autoencoder(input_dim, config["hidden_dims"], config["bottleneck_dim"], config["activation"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.MSELoss()

    fit_tensor = torch.tensor(X_fit, dtype=torch.float32).to(device)
    internal_val_tensor = torch.tensor(X_internal_val, dtype=torch.float32).to(device)
    n_fit = fit_tensor.shape[0]
    batch_size = config["batch_size"]

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(config["epochs"]):
        model.train()
        perm = torch.randperm(n_fit, device=device)
        batch_losses = []
        for start in range(0, n_fit, batch_size):
            batch = fit_tensor[perm[start:start + batch_size]]
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(internal_val_tensor), internal_val_tensor).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose:
            print(f"  Epoch {epoch + 1:3d}/{config['epochs']}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config["early_stopping_patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch + 1} (best val_loss={best_val_loss:.6f})")
                break

    model.load_state_dict(best_state)
    return model, history

### 7.2 Train BENIGN-only, on the actual (reduced) BENIGN population documented in Section 5

In [12]:
benign_fit_df, benign_internal_val_df = train_test_split(
    behavioral_train_benign_df, test_size=BENIGN_INTERNAL_VAL_FRAC, random_state=RANDOM_STATE
)
print(f"BENIGN fit rows: {len(benign_fit_df):,}  (of {n_behavioral_benign:,} total BENIGN train rows, "
      f"{BENIGN_INTERNAL_VAL_FRAC:.0%} held out below as internal_val)")
print(f"BENIGN internal_val rows: {len(benign_internal_val_df):,}")

X_benign_fit = benign_fit_df[feature_cols].to_numpy(dtype=np.float32)
X_benign_internal_val = benign_internal_val_df[feature_cols].to_numpy(dtype=np.float32)

_train_start = time.perf_counter()
ae_model, ae_history = train_autoencoder(
    X_benign_fit, X_benign_internal_val, input_dim=len(feature_cols), config=AE_CONFIG, device=DEVICE, verbose=True
)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
train_seconds = time.perf_counter() - _train_start
print(f"\nTrained for {len(ae_history['train_loss'])} epochs. Wall-clock: {train_seconds:.2f}s (device={DEVICE})")

BENIGN fit rows: 864,653  (of 960,726 total BENIGN train rows, 10% held out below as internal_val)
BENIGN internal_val rows: 96,073


  Epoch   1/30  train_loss=0.029807  val_loss=0.012058
  Epoch   2/30  train_loss=0.007979  val_loss=0.004247


  Epoch   3/30  train_loss=0.003036  val_loss=0.002276
  Epoch   4/30  train_loss=0.001843  val_loss=0.001377


  Epoch   5/30  train_loss=0.001199  val_loss=0.001018
  Epoch   6/30  train_loss=0.000901  val_loss=0.000756


  Epoch   7/30  train_loss=0.000673  val_loss=0.000574
  Epoch   8/30  train_loss=0.000516  val_loss=0.000439


  Epoch   9/30  train_loss=0.000401  val_loss=0.000354
  Epoch  10/30  train_loss=0.000340  val_loss=0.000313


... (8 outputs left out)

  Epoch  25/30  train_loss=0.000160  val_loss=0.000154
  Epoch  26/30  train_loss=0.000154  val_loss=0.000147


  Epoch  27/30  train_loss=0.000147  val_loss=0.000139


  Epoch  28/30  train_loss=0.000143  val_loss=0.000133
  Epoch  29/30  train_loss=0.000136  val_loss=0.000129


  Epoch  30/30  train_loss=0.000131  val_loss=0.000127

Trained for 30 epochs. Wall-clock: 6.96s (device=cuda)


### 7.3 Threshold sweep -- same percentile grid as `03_autoencoder.ipynb`

In [13]:
X_val_ae = val_df[feature_cols].to_numpy(dtype=np.float32)
val_errors = compute_reconstruction_error(ae_model, X_val_ae, DEVICE)
val_df["ae_reconstruction_error"] = val_errors
is_attack = (val_df["Label"] != "BENIGN").astype(int).to_numpy()

benign_internal_val_errors = compute_reconstruction_error(ae_model, X_benign_internal_val, DEVICE)
auc_roc_ae = roc_auc_score(is_attack, val_errors)

sweep_rows_ae = []
for pct in THRESHOLD_PERCENTILES:
    threshold = float(np.percentile(benign_internal_val_errors, pct))
    pred = (val_errors > threshold).astype(int)
    sweep_rows_ae.append({
        "percentile": pct,
        "threshold": threshold,
        "precision": precision_score(is_attack, pred),
        "recall": recall_score(is_attack, pred),
        "f1": f1_score(is_attack, pred),
        "auc_roc": auc_roc_ae,
    })

ae_sweep_df = pd.DataFrame(sweep_rows_ae)
ae_sweep_df

,percentile,threshold,precision,recall,f1,auc_roc
0,90,0.000158,0.706484,0.929992,0.802975,0.979323
1,95,0.000383,0.823003,0.912400,0.865399,0.979323
2,99,0.002148,0.954196,0.831834,0.888823,0.979323


In [14]:
best_row_ae = ae_sweep_df.loc[ae_sweep_df["f1"].idxmax()]
AE_CHOSEN_THRESHOLD = float(best_row_ae["threshold"])
AE_CHOSEN_PERCENTILE = int(best_row_ae["percentile"])
print(f"Chosen threshold: {AE_CHOSEN_THRESHOLD:.6f}  ({AE_CHOSEN_PERCENTILE}th percentile of BENIGN internal-val reconstruction error)  F1={best_row_ae['f1']:.4f}")

ae_prediction = (val_errors > AE_CHOSEN_THRESHOLD).astype(int)
val_df["ae_prediction"] = ae_prediction

ae_precision = precision_score(is_attack, ae_prediction)
ae_recall = recall_score(is_attack, ae_prediction)
ae_f1 = f1_score(is_attack, ae_prediction)

print(f"\nFinal (chosen threshold) -- Precision {ae_precision:.4f}  Recall {ae_recall:.4f}  F1 {ae_f1:.4f}  AUC-ROC {auc_roc_ae:.4f}")

Chosen threshold: 0.002148  (99th percentile of BENIGN internal-val reconstruction error)  F1=0.8888

Final (chosen threshold) -- Precision 0.9542  Recall 0.8318  F1 0.8888  AUC-ROC 0.9793


## 7.4 Per-category recall (Autoencoder)

In [15]:
ae_category_recall = (
    val_df[val_df["Label"] != "BENIGN"]
    .groupby("Label")
    .apply(lambda g: pd.Series({"recall": g["ae_prediction"].mean(), "n_rows": len(g)}))
    .sort_values("recall")
)
ae_category_recall

C:\Users\devMrX\AppData\Local\Temp\ipykernel_17820\3655952766.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"recall": g["ae_prediction"].mean(), "n_rows": len(g)}))


,recall,n_rows
Label,,
SSH-Patator,0.000000,644.0
Web Attack - Brute Force,0.000000,294.0
Web Attack - Sql Injection,0.000000,4.0
Web Attack - XSS,0.000000,131.0
FTP-Patator,0.000843,1186.0
Bot,0.023136,389.0
Infiltration,0.142857,7.0
DoS Slowhttptest,0.266029,1045.0
DoS GoldenEye,0.342246,2057.0


## 8. Save models + reports (namespaced `_behavioral_v2` -- distinct from the baseline and from `04_enhanced_ablation_behavioral.ipynb`'s standalone prototype)

In [16]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if_model_path = MODELS_DIR / "isolation_forest_behavioral_v2.pkl"
joblib.dump(iso_forest_main, if_model_path)
print("Saved:", if_model_path)

ae_model_path = MODELS_DIR / "autoencoder_behavioral_v2.pt"
torch.save(ae_model.state_dict(), ae_model_path)
print("Saved:", ae_model_path)

ae_config_to_save = dict(AE_CONFIG)
ae_config_to_save.update({
    "input_dim": len(feature_cols),
    "chosen_threshold": AE_CHOSEN_THRESHOLD,
    "chosen_percentile": AE_CHOSEN_PERCENTILE,
    "random_state": RANDOM_STATE,
    "benign_internal_val_frac": BENIGN_INTERNAL_VAL_FRAC,
    "epochs_run": len(ae_history["train_loss"]),
    "train_benign_row_count": n_behavioral_benign,
    "baseline_train_benign_row_count": BASELINE_TRAIN_BENIGN_COUNT,
    "benign_row_count_reduction_pct": pct_reduction,
})
ae_config_path = MODELS_DIR / "autoencoder_behavioral_v2_config.json"
with open(ae_config_path, "w", encoding="utf-8") as f:
    json.dump(ae_config_to_save, f, indent=2)
print("Saved:", ae_config_path)

Saved: E:\thesis\models\isolation_forest_behavioral_v2.pkl
Saved: E:\thesis\models\autoencoder_behavioral_v2.pt
Saved: E:\thesis\models\autoencoder_behavioral_v2_config.json


In [17]:
val_metrics_path = REPORTS_DIR / "behavioral_v2_val_metrics.csv"
with open(val_metrics_path, "w", encoding="utf-8") as f:
    f.write("# BENIGN training population (Autoencoder) -- actual row counts, not just a label\n")
    pd.DataFrame([{
        "baseline_80f_train_benign_rows": BASELINE_TRAIN_BENIGN_COUNT,
        "behavioral_v2_train_benign_rows": n_behavioral_benign,
        "reduction_pct": pct_reduction,
    }]).to_csv(f, index=False)

    f.write("\n# Isolation Forest -- contamination sweep\n")
    contamination_sweep_df.to_csv(f, index=False)
    f.write(f"\n# Isolation Forest -- chosen contamination={CONTAMINATION}, final val metrics\n")
    pd.DataFrame([{
        "model": "IsolationForest", "n_features": len(feature_cols),
        "precision": if_precision, "recall": if_recall, "f1": if_f1, "auc_roc": if_auc_roc,
    }]).to_csv(f, index=False)
    f.write("\n# Isolation Forest -- per-category recall (attack classes only)\n")
    if_category_recall.reset_index().to_csv(f, index=False)

    f.write("\n# Autoencoder -- threshold sweep\n")
    ae_sweep_df.to_csv(f, index=False)
    f.write(f"\n# Autoencoder -- chosen percentile={AE_CHOSEN_PERCENTILE}, threshold={AE_CHOSEN_THRESHOLD}, final val metrics\n")
    pd.DataFrame([{
        "model": "Autoencoder", "n_features": len(feature_cols),
        "precision": ae_precision, "recall": ae_recall, "f1": ae_f1, "auc_roc": auc_roc_ae,
    }]).to_csv(f, index=False)
    f.write("\n# Autoencoder -- per-category recall (attack classes only)\n")
    ae_category_recall.reset_index().to_csv(f, index=False)

print("Saved:", val_metrics_path)

Saved: E:\thesis\reports\behavioral_v2_val_metrics.csv


## 9. Comparison: baseline (80-feature) vs. standalone prototype (`04_enhanced_ablation_behavioral.ipynb`) vs. this run (properly-integrated behavioral_v2)

This is the actual decision-relevant check: does the properly pipeline-integrated version
(Phase 1 row-hash-join-validated, Phase 2 impute/CTI-encode/scale/SMOTE-migrated) regress
relative to the earlier standalone notebook's quick, unvalidated prototype split? If not, Option A
is confirmed safe to commit to for real.

In [18]:
baseline_if_row = pd.read_csv(REPORTS_DIR / "isolation_forest_val_metrics.csv")
baseline_if_row = baseline_if_row[baseline_if_row["status"] == "current_contamination_tuned"].iloc[0]

# autoencoder_val_metrics.csv mixes multiple labeled sections with blank-line separators (and
# CRLF line endings) -- parse robustly by locating the marker line then taking the next two
# non-blank lines (header, data), rather than assuming a fixed blank-line-delimited layout.
with open(REPORTS_DIR / "autoencoder_val_metrics.csv", newline="") as f:
    ae_lines = [line.strip() for line in f.read().splitlines()]
marker_idx = ae_lines.index("# final metrics at chosen threshold")
non_blank_after = [line for line in ae_lines[marker_idx + 1:] if line]
baseline_ae_header, baseline_ae_data = non_blank_after[0], non_blank_after[1]
baseline_ae_final = pd.read_csv(pd.io.common.StringIO(baseline_ae_header + "\n" + baseline_ae_data)).iloc[0]

enhanced_ablation_df = pd.read_csv(REPORTS_DIR / "enhanced_ablation_val_metrics.csv").set_index(["model", "feature_set"])

comparison_rows = [
    {"model": "IsolationForest", "variant": "baseline_80f (main pipeline)",
     "n_features": int(baseline_if_row["n_features"]), "precision": baseline_if_row["precision"],
     "recall": baseline_if_row["recall"], "f1": baseline_if_row["f1"], "auc_roc": baseline_if_row["auc_roc"]},
    {"model": "IsolationForest", "variant": "enhanced_ablation_92f (standalone prototype)",
     "n_features": int(enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "n_features"]),
     "precision": enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "precision"],
     "recall": enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "recall"],
     "f1": enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "f1"],
     "auc_roc": enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "auc_roc"]},
    {"model": "IsolationForest", "variant": "behavioral_v2_92f (Phase 1+2 integrated, this run)",
     "n_features": len(feature_cols), "precision": if_precision, "recall": if_recall, "f1": if_f1, "auc_roc": if_auc_roc},

    {"model": "Autoencoder", "variant": "baseline_80f (main pipeline)",
     "n_features": 80, "precision": baseline_ae_final["precision"],
     "recall": baseline_ae_final["recall"], "f1": baseline_ae_final["f1"], "auc_roc": baseline_ae_final["auc_roc"]},
    {"model": "Autoencoder", "variant": "enhanced_ablation_92f (standalone prototype)",
     "n_features": int(enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "n_features"]),
     "precision": enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "precision"],
     "recall": enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "recall"],
     "f1": enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "f1"],
     "auc_roc": enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "auc_roc"]},
    {"model": "Autoencoder", "variant": "behavioral_v2_92f (Phase 1+2 integrated, this run)",
     "n_features": len(feature_cols), "precision": ae_precision, "recall": ae_recall, "f1": ae_f1, "auc_roc": auc_roc_ae},
]

three_way_comparison_df = pd.DataFrame(comparison_rows)
comparison_path = REPORTS_DIR / "behavioral_v2_vs_baseline_comparison.csv"
three_way_comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)
three_way_comparison_df

Saved: E:\thesis\reports\behavioral_v2_vs_baseline_comparison.csv


,model,variant,n_features,precision,recall,f1,auc_roc
0,IsolationForest,baseline_80f (main pipeline),80,0.529191,0.555785,0.542162,0.728624
1,IsolationForest,enhanced_ablation_92f (standalone prototype),92,0.516811,0.613050,0.560832,0.863762
2,IsolationForest,"behavioral_v2_92f (Phase 1+2 integrated, this ...",92,0.586964,0.554082,0.570049,0.837760
3,Autoencoder,baseline_80f (main pipeline),80,0.913755,0.521480,0.664010,0.864104
4,Autoencoder,enhanced_ablation_92f (standalone prototype),92,0.947213,0.883215,0.914095,0.984377
5,Autoencoder,"behavioral_v2_92f (Phase 1+2 integrated, this ...",92,0.954196,0.831834,0.888823,0.979323


## 10. Conclusion

(Synthesized from the actual run results above.)

In [19]:
print("=== Isolation Forest ===")
if_rows = three_way_comparison_df[three_way_comparison_df["model"] == "IsolationForest"]
print(if_rows.to_string(index=False))
print()
print("=== Autoencoder ===")
ae_rows = three_way_comparison_df[three_way_comparison_df["model"] == "Autoencoder"]
print(ae_rows.to_string(index=False))

print()
if_regression = if_f1 < enhanced_ablation_df.loc[("IsolationForest", "behavioral_92f"), "f1"]
ae_regression = ae_f1 < enhanced_ablation_df.loc[("Autoencoder", "behavioral_92f"), "f1"]
print(f"IsolationForest F1 regressed vs. standalone prototype: {if_regression}  "
      f"({if_f1:.4f} vs. {enhanced_ablation_df.loc[('IsolationForest', 'behavioral_92f'), 'f1']:.4f})")
print(f"Autoencoder F1 regressed vs. standalone prototype: {ae_regression}  "
      f"({ae_f1:.4f} vs. {enhanced_ablation_df.loc[('Autoencoder', 'behavioral_92f'), 'f1']:.4f})")

=== Isolation Forest ===
          model                                            variant  n_features  precision   recall       f1  auc_roc
IsolationForest                       baseline_80f (main pipeline)          80   0.529191 0.555785 0.542162 0.728624
IsolationForest       enhanced_ablation_92f (standalone prototype)          92   0.516811 0.613050 0.560832 0.863762
IsolationForest behavioral_v2_92f (Phase 1+2 integrated, this run)          92   0.586964 0.554082 0.570049 0.837760

=== Autoencoder ===
      model                                            variant  n_features  precision   recall       f1  auc_roc
Autoencoder                       baseline_80f (main pipeline)          80   0.913755 0.521480 0.664010 0.864104
Autoencoder       enhanced_ablation_92f (standalone prototype)          92   0.947213 0.883215 0.914095 0.984377
Autoencoder behavioral_v2_92f (Phase 1+2 integrated, this run)          92   0.954196 0.831834 0.888823 0.979323

IsolationForest F1 regressed vs. 